# RAG OpenAI Assistants in AG2

````{=mdx}
:::warning
`GPTAssistantAgent` is deprecated as of v0.12 and will be removed in v0.14. Use `ConversableAgent` instead. This notebook will also be removed in v0.14.
:::
````

This notebook shows an example of the [`GPTAssistantAgent`](https://github.com/ag2ai/ag2/blob/main/autogen/agentchat/contrib/gpt_assistant_agent.py) with retrieval augmented generation. `GPTAssistantAgent` is an experimental AG2 agent class that leverages the [OpenAI Assistant API](https://platform.openai.com/docs/assistants/overview) for conversational capabilities,  working with
`UserProxyAgent` in AG2.

In [ ]:
import logging
import os

import autogen
from autogen import UserProxyAgent
from autogen.agentchat.contrib.gpt_assistant_agent import GPTAssistantAgent

logger = logging.getLogger(__name__)
logger.setLevel(logging.WARNING)

assistant_id = os.environ.get("ASSISTANT_ID", None)

llm_config = autogen.LLMConfig(
    config_list=[
        {
            "model": "gpt-5",
            "api_key": os.getenv("OPENAI_API_KEY"),
            "api_type": "openai",
        },
        {
            "model": "gpt-5-mini",
            "api_key": os.getenv("OPENAI_API_KEY"),
            "api_type": "openai",
        },
    ]
)
assistant_config = {
    "assistant_id": assistant_id,
    "tools": [{"type": "retrieval"}],
    "file_ids": ["file-AcnBk5PCwAjJMCVO0zLSbzKP"],
    # add id of an existing file in your openai account
    # in this case I added the implementation of conversable_agent.py
}

gpt_assistant = GPTAssistantAgent(
    name="assistant",
    instructions="You are adapt at question answering",
    llm_config=llm_config,
    assistant_config=assistant_config,
)

user_proxy = UserProxyAgent(
    name="user_proxy",
    code_execution_config=False,
    is_termination_msg=lambda msg: "TERMINATE" in msg["content"],
    human_input_mode="ALWAYS",
)
user_proxy.initiate_chat(gpt_assistant, message="Please explain the code in this file!")

gpt_assistant.delete_assistant()